# Test Pipeline Inspection Results
*Co-authored with CoCo*

Cell-by-cell inspection of the TEST_PIPELINE_DB pipeline state.
Run after `01_test_deploy.sql` → `02_test_seed_and_run.sql` to validate each layer.

In [ ]:
%%sql -r ctx
USE WAREHOUSE COMPUTE_WH;
USE DATABASE TEST_PIPELINE_DB;

---
## SEED LAYER
Validate that seed data was loaded correctly and SIMULATE_DAILY_LOAD state is tracking.

In [ ]:
%%sql -r seed_row_count
-- Seed table row count
SELECT 'SEED_MATCHES_SUMMARY' AS TABLE_NAME, COUNT(*) AS ROW_COUNT
FROM SEED.SEED_MATCHES_SUMMARY;

In [ ]:
-- Current load state: where is the simulation pointer?
SELECT
    CURRENT_LOAD_DATE,
    MIN_DATE,
    MAX_DATE,
    LAST_LOADED_AT,
    DATEDIFF('day', MIN_DATE, CURRENT_LOAD_DATE) AS DAYS_REMAINING
FROM SEED.SEED_LOAD_STATE;

In [ ]:
%%sql -r seed_date_dist
-- Date distribution in seed: how many matches per date?
SELECT
    GAME_DATE_DAY,
    COUNT(*) AS MATCH_COUNT
FROM SEED.SEED_MATCH_DATE_INDEX
GROUP BY GAME_DATE_DAY
ORDER BY GAME_DATE_DAY DESC
LIMIT 20;

In [ ]:
%%sql -r next_batch
-- Next simulation batch preview: how many rows would the next SIMULATE_DAILY_LOAD stage?
SELECT
    s.CURRENT_LOAD_DATE AS NEXT_LOAD_DATE,
    COUNT(d.MATCH_ID) AS MATCHES_IN_NEXT_BATCH
FROM SEED.SEED_LOAD_STATE s
LEFT JOIN SEED.SEED_MATCH_DATE_INDEX d
    ON d.GAME_DATE_DAY = s.CURRENT_LOAD_DATE
GROUP BY s.CURRENT_LOAD_DATE;

---
## BRONZE LAYER
Validate ingestion into bronze: staged files, pipe history, table contents, stream state.

In [ ]:
%%sql -r bronze_count
-- Bronze table row count
SELECT COUNT(*) AS BRONZE_ROW_COUNT
FROM BRONZE.MATCHES_SUMMARY_BRONZE;

In [ ]:
%%sql -r bronze_sample
-- Sample bronze rows (most recent by LDTS)
SELECT *
FROM BRONZE.MATCHES_SUMMARY_BRONZE
ORDER BY LDTS DESC
LIMIT 10;

In [ ]:
%%sql -r stage_placeholder
-- Stage file listing: what files are in the matches stage?
SELECT *
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
WHERE 1=0; -- placeholder
-- Use LIST instead:
-- (LIST is not queryable in a notebook cell, so we check via pipe history below)

In [ ]:
%%sql -r copy_history
-- Copy history for matches summary pipe
SELECT
    FILE_NAME,
    STAGE_LOCATION,
    STATUS,
    ROW_COUNT,
    ROW_PARSED,
    FIRST_ERROR_MESSAGE,
    LAST_LOAD_TIME
FROM TABLE(INFORMATION_SCHEMA.COPY_HISTORY(
    TABLE_NAME => 'BRONZE.MATCHES_SUMMARY_BRONZE',
    START_TIME => DATEADD('hour', -24, CURRENT_TIMESTAMP())
))
ORDER BY LAST_LOAD_TIME DESC;

In [ ]:
%%sql -r stream_state
-- Stream state: does the bronze stream have unconsumed rows?
SELECT
    SYSTEM$STREAM_HAS_DATA('BRONZE.MATCHES_SUMMARY_BRONZE_STM') AS STREAM_HAS_DATA;

---
## SILVER LAYER
Validate that the task consumed the stream and merged data into the silver table.

In [ ]:
%%sql -r silver_count
-- Silver table row count
SELECT COUNT(*) AS SILVER_ROW_COUNT
FROM SILVER.MATCHES_SUMMARY_SILVER;

In [ ]:
%%sql -r silver_sample
-- Sample silver rows
SELECT *
FROM SILVER.MATCHES_SUMMARY_SILVER
ORDER BY GAME_DATE DESC NULLS LAST
LIMIT 10;

In [ ]:
%%sql -r task_history
-- Task execution history (last 24h)
SELECT
    NAME,
    STATE,
    QUERY_START_TIME,
    COMPLETED_TIME,
    DATEDIFF('second', QUERY_START_TIME, COMPLETED_TIME) AS DURATION_SEC,
    ERROR_CODE,
    ERROR_MESSAGE,
    RETURN_VALUE
FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(
    TASK_NAME => 'BRONZE_TO_SILVER_MATCHES_TASK',
    SCHEDULED_TIME_RANGE_START => DATEADD('hour', -24, CURRENT_TIMESTAMP()),
    SCHEDULED_TIME_RANGE_END => CURRENT_TIMESTAMP()
))
ORDER BY QUERY_START_TIME DESC;

In [ ]:
%%sql -r task_state
-- Task state: is the task currently running/suspended?
SHOW TASKS LIKE 'BRONZE_TO_SILVER_MATCHES_TASK' IN SCHEMA SILVER;

In [ ]:
%%sql -r stream_consumed
-- Stream offset check: has the stream been fully consumed?
SELECT
    SYSTEM$STREAM_HAS_DATA('BRONZE.MATCHES_SUMMARY_BRONZE_STM') AS STREAM_HAS_UNCONSUMED_DATA;

---
## CROSS-LAYER RECONCILIATION
Compare row counts across layers to detect data loss or duplicates.

In [ ]:
%%sql -r reconciliation
-- Row count reconciliation: seed vs bronze vs silver
SELECT
    'SEED' AS LAYER, COUNT(*) AS ROW_COUNT FROM SEED.SEED_MATCHES_SUMMARY
UNION ALL
SELECT
    'BRONZE', COUNT(*) FROM BRONZE.MATCHES_SUMMARY_BRONZE
UNION ALL
SELECT
    'SILVER', COUNT(*) FROM SILVER.MATCHES_SUMMARY_SILVER;

In [ ]:
%%sql -r freshness
-- Data freshness: latest timestamps at each layer
SELECT
    'BRONZE' AS LAYER,
    MAX(LDTS) AS LATEST_TIMESTAMP
FROM BRONZE.MATCHES_SUMMARY_BRONZE
UNION ALL
SELECT
    'SILVER',
    MAX(GAME_DATE)
FROM SILVER.MATCHES_SUMMARY_SILVER;

In [ ]:
%%sql -r orphan_check
-- Orphan check: bronze rows not yet in silver (stream lag)
SELECT COUNT(*) AS BRONZE_NOT_IN_SILVER
FROM BRONZE.MATCHES_SUMMARY_BRONZE b
WHERE NOT EXISTS (
    SELECT 1 FROM SILVER.MATCHES_SUMMARY_SILVER s
    WHERE s.MATCH_ID = UPPER(TRIM(b.MATCH_ID))
);

In [ ]:
%%sql -r duplicates
-- Duplicate check: any MATCH_ID appearing more than once in silver?
SELECT MATCH_ID, COUNT(*) AS OCCURRENCES
FROM SILVER.MATCHES_SUMMARY_SILVER
GROUP BY MATCH_ID
HAVING COUNT(*) > 1
LIMIT 10;